In [20]:
import pandas as pd

with open(r"C:\Users\sebal\Downloads\BI_T-main\GRD\GRD_PUBLICO_EXTERNO_2022.txt", "r", encoding="utf-16") as f:
    print(repr(f.readline()))  # confirm it reads cleanly now

df_ext = pd.read_csv(
    r"C:\Users\sebal\Downloads\BI_T-main\GRD\GRD_PUBLICO_EXTERNO_2022.txt",
    sep="|", encoding="utf-16", low_memory=False
)
print(df_ext.shape)

'COD_HOSPITAL|CIP_ENCRIPTADO|SEXO|FECHA_NACIMIENTO|ETNIA|PROVINCIA|COMUNA|NACIONALIDAD|PREVISION|SERVICIO_SALUD|TIPO_PROCEDENCIA|TIPO_INGRESO|ESPECIALIDAD_MEDICA|TIPO_ACTIVIDAD|FECHA_INGRESO|SERVICIOINGRESO|FECHATRASLADO1|SERVICIOTRASLADO1|FECHATRASLADO2|SERVICIOTRASLADO2|FECHATRASLADO3|SERVICIOTRASLADO3|FECHATRASLADO4|SERVICIOTRASLADO4|FECHATRASLADO5|SERVICIOTRASLADO5|FECHATRASLADO6|SERVICIOTRASLADO6|FECHATRASLADO7|SERVICIOTRASLADO7|FECHATRASLADO8|SERVICIOTRASLADO8|FECHATRASLADO9|SERVICIOTRASLADO9|FECHAALTA|SERVICIOALTA|TIPOALTA|CONDICIONDEALTANEONATO1|PESORN1|SEXORN1|RN1ESTADO|CONDICIONDEALTANEONATO2|PESORN2|SEXORN2|RN2ESTADO|CONDICIONDEALTANEONATO3|PESORN3|SEXORN3|RN3ESTADO|CONDICIONDEALTANEONATO4|PESORN4|SEXORN4|RN4ESTADO|DIAGNOSTICO1|DIAGNOSTICO2|DIAGNOSTICO3|DIAGNOSTICO4|DIAGNOSTICO5|DIAGNOSTICO6|DIAGNOSTICO7|DIAGNOSTICO8|DIAGNOSTICO9|DIAGNOSTICO10|DIAGNOSTICO11|DIAGNOSTICO12|DIAGNOSTICO13|DIAGNOSTICO14|DIAGNOSTICO15|DIAGNOSTICO16|DIAGNOSTICO17|DIAGNOSTICO18|DIAGNOSTICO19|DIAGNOS

ParserError: Error tokenizing data. C error: Expected 129 fields in line 120127, saw 130


In [ ]:
path = r"C:\Users\sebal\Downloads\BI_T-main\GRD\GRD_PUBLICO_EXTERNO_2022.txt"

with open(path, "r", encoding="utf-16") as f:
    header = f.readline()
    expected_pipes = header.count("|")  # pipes in header = n_columns - 1

    bad_lines = []
    for i, line in enumerate(f, start=2):  # start=2: line 1 is header
        n = line.count("|")
        if n != expected_pipes:
            bad_lines.append((i, n))

print(f"Header has {expected_pipes} separators (i.e. {expected_pipes + 1} columns)")
print(f"Found {len(bad_lines)} malformed lines out of file")
print(bad_lines[:10])  # just line numbers + pipe counts, no patient data

Header has 128 separators (i.e. 129 columns)
Found 1 malformed lines out of file
[(120127, 129)]


In [ ]:
df_ext = pd.read_csv(
    r"C:\Users\sebal\Downloads\BI_T-main\GRD\GRD_PUBLICO_EXTERNO_2022.txt",
    sep="|", encoding="utf-16",
    engine="python", on_bad_lines="skip"
)
print(df_ext.shape)
print("Rows skipped due to malformed delimiters: 1 (line 120127 of the raw file)")

(932839, 129)
Rows skipped due to malformed delimiters: 1 (line 120127 of the raw file)


In [ ]:
print(df_ext["PREVISION"].value_counts(dropna=False))

PREVISION
FONASA INSTITUCIONAL - (MAI) B    439655
FONASA INSTITUCIONAL - (MAI) A    217676
FONASA INSTITUCIONAL - (MAI) D    129793
FONASA INSTITUCIONAL - (MAI) C    109295
FONASA LIBRE ELECCIÓN (FMLE_B)      9304
PARTICULAR                          7553
ISAPRE                              7471
FONASA LIBRE ELECCIÓN (FMLE_D)      5781
FONASA LIBRE ELECCIÓN (FMLE_C)      2782
DIPRECA                             2228
CAPREDENA                            704
CAJA DE PREVISIÓN FFAA (SISA)        538
DESCONOCIDO                           59
Name: count, dtype: int64


We explored the datasets, specially 2022 as it seems irregular. Aside from a different formating, it seems to be the same overall data so it will be included as part of the public GRD data.

In [1]:
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # remove if you want inline plots
import matplotlib.pyplot as plt

In [2]:
# Cell — paths and encodings
# 2022 is published as "EXTERNO" instead of "PUBLICO" and is UTF-16 encoded,
# unlike the other three years (latin1). Confirmed via header inspection.
GRD_PATHS = {
    2021: (r"C:\Users\sebal\Downloads\BI_T-main\GRD\GRD_PUBLICO_2021.txt", "latin1"),
    2022: (r"C:\Users\sebal\Downloads\BI_T-main\GRD\GRD_PUBLICO_EXTERNO_2022.txt", "utf-16"),
    2023: (r"C:\Users\sebal\Downloads\BI_T-main\GRD\GRD_PUBLICO_2023.txt", "utf-16"),
    2024: (r"C:\Users\sebal\Downloads\BI_T-main\GRD\GRD_PUBLICO_2024.txt", "latin1"),
}

In [3]:
# Cell — insurance grouping map (validated against 2024 data)
INSURANCE_MAP = {
    "FONASA INSTITUCIONAL - (MAI) A": "FONASA", "FONASA INSTITUCIONAL - (MAI) B": "FONASA",
    "FONASA INSTITUCIONAL - (MAI) C": "FONASA", "FONASA INSTITUCIONAL - (MAI) D": "FONASA",
    "FONASA LIBRE ELECCIÓN (FMLE_B)": "FONASA", "FONASA LIBRE ELECCIÓN (FMLE_C)": "FONASA",
    "FONASA LIBRE ELECCIÓN (FMLE_D)": "FONASA",
    "ISAPRE": "PRIVATE", "PARTICULAR": "PRIVATE",
    "DIPRECA": "MILITARY", "CAPREDENA": "MILITARY", "CAJA DE PREVISIÓN FFAA (SISA)": "MILITARY",
    "DESCONOCIDO": None,
}

# Cell — columns actually needed (skips the other ~125 columns for speed)
NEEDED_COLS = ["PREVISION", "IR_29301_PESO", "IR_29301_SEVERIDAD", "IR_29301_MORTALIDAD"]

In [4]:
# Cell — load + clean function
def load_and_clean_grd(year, path, encoding):
    """
    Loads one year of GRD data and applies the validated cleaning steps:
    - parse PESO (comma decimal -> float)
    - group PREVISION into FONASA / PRIVATE / MILITARY
    - exclude ambulatory-surgery records with no real clinical grouping
      (SEVERIDAD == '0' and MORTALIDAD == '0', confirmed to be ~100% CMA cases in 2024)
    - handle the single malformed line found in the 2022 (UTF-16) file
    Returns the cleaned dataframe plus a stats dict for the Process record.
    """
    is_utf16 = (encoding == "utf-16")

    read_kwargs = dict(sep="|", encoding=encoding, on_bad_lines="skip", usecols=NEEDED_COLS)
    if is_utf16:
        read_kwargs["engine"] = "python"
    else:
        read_kwargs["engine"] = "c"
        read_kwargs["low_memory"] = False

    df = pd.read_csv(path, **read_kwargs)
    n_raw = len(df)

    df["PESO_num"] = pd.to_numeric(
        df["IR_29301_PESO"].astype(str).str.replace(",", ".", regex=False), errors="coerce"
    )
    df["insurance_group"] = df["PREVISION"].map(INSURANCE_MAP)

    no_grouping = (df["IR_29301_SEVERIDAD"].astype(str) == "0") & \
                  (df["IR_29301_MORTALIDAD"].astype(str) == "0")
    n_no_grouping = no_grouping.sum()

    valid = df[df["insurance_group"].notna() & ~no_grouping &
               (df["IR_29301_SEVERIDAD"].astype(str) != "DESCONOCIDO")].copy()
    valid["high_severity"] = valid["IR_29301_SEVERIDAD"].astype(str) == "3"

    stats = {
        "year": year, "n_raw": n_raw,
        "n_unknown_insurance": df["insurance_group"].isna().sum(),
        "n_no_clinical_grouping": n_no_grouping,
        "n_valid": len(valid),
    }
    return valid, stats

In [ ]:
# Cell — run for all 4 years, with progress prints
print("Starting load for:", list(GRD_PATHS.keys()))

results = []
stats_log = []

for year, (path, enc) in GRD_PATHS.items():
    print(f"  loading {year} ({enc})...")
    valid, stats = load_and_clean_grd(year, path, enc)
    stats_log.append(stats)
    print(f"  {year} done — {stats['n_valid']} valid rows (of {stats['n_raw']} raw)")

    kpi1 = valid.groupby("insurance_group")["PESO_num"].mean()
    kpi2 = valid.groupby("insurance_group")["high_severity"].mean() * 100
    n = valid["insurance_group"].value_counts()

    results.append({
        "year": year,
        "avg_weight_FONASA": kpi1.get("FONASA"), "avg_weight_PRIVATE": kpi1.get("PRIVATE"),
        "pct_high_sev_FONASA": kpi2.get("FONASA"), "pct_high_sev_PRIVATE": kpi2.get("PRIVATE"),
        "gap_pp": kpi2.get("FONASA", 0) - kpi2.get("PRIVATE", 0),
        "n_FONASA": n.get("FONASA", 0), "n_PRIVATE": n.get("PRIVATE", 0),
    })

summary = pd.DataFrame(results).sort_values("year")
stats_df = pd.DataFrame(stats_log).sort_values("year")

print("\n=== SUMMARY ===")
print(summary)
print()
print(stats_df)

Starting load for: [2021, 2022, 2023, 2024]
  loading 2021 (latin1)...
  2021 done — 703387 valid rows (of 816909 raw)
  loading 2022 (utf-16)...
  2022 done — 777989 valid rows (of 932840 raw)
  loading 2023 (utf-16)...
  2023 done — 845984 valid rows (of 1039587 raw)
  loading 2024 (latin1)...
  2024 done — 873711 valid rows (of 1085813 raw)

=== SUMMARY ===
   year  avg_weight_FONASA  avg_weight_PRIVATE  pct_high_sev_FONASA  \
0  2021           1.189595            1.261929            26.920129   
1  2022           1.059461            0.986660            24.682114   
2  2023           1.054729            1.000686            24.194209   
3  2024           1.067676            1.044792            24.954563   

   pct_high_sev_PRIVATE    gap_pp  n_FONASA  n_PRIVATE  
0             28.698315 -1.778186    687623      13060  
1             19.333863  5.348251    760887      13841  
2             19.330575  4.863634    831947      10636  
3             19.454653  5.499910    861638       843

In [ ]:
for year, (path, enc) in GRD_PATHS.items():
    if enc == "latin1":
        with open(path, "r", encoding="latin1") as f:
            header = f.readline()
        cols = header.strip().split("|")
        print(f"--- {year} ---")
        print("n columns:", len(cols))
        missing = [c for c in NEEDED_COLS if c not in cols]
        print("missing from NEEDED_COLS:", missing if missing else "none — all found")
        print()

--- 2021 ---
n columns: 129
missing from NEEDED_COLS: none — all found

--- 2024 ---
n columns: 129
missing from NEEDED_COLS: none — all found



2021 shows an inverse pattern (PRIVATE severity > FONASA severity, gap of -1.8pp) compared to 2022-2024. This coincides with a markedly lower share of ambulatory-surgery cases with no clinical grouping (102,938 vs. 150-212K in other years), consistent with elective surgeries being postponed during the peak pandemic year, leaving a higher proportion of genuine hospitalizations with assigned severity, a plausible contextual explanation rather than a data defect.

2021 also shows an unusually high count of unmapped PREVISION categories (11,601 vs. ~50 in other years). Noted as an open observation, not further investigated given time constraints.

In [ ]:
# ---------------------------------------------------------------------------
# RESEARCH QUESTION - REFINEMENT NOTE
# ---------------------------------------------------------------------------
# Original framing (W1): compare complexity between public and private
# hospitals. Rejected once we confirmed that Chile's open GRD data only
# covers the public network (SNSS) - private hospitals are not required to
# publish case-mix data. See W1 report for that decision.
#
# First C1 framing: within the public network, do FONASA beneficiaries show
# higher severity than PRIVATE (Isapre/Particular) beneficiaries?
#
# Refinement (this note): patients with private insurance who appear in this
# dataset are a self-selected subsample - they have private coverage but
# still ended up in a public hospital anyway (likely due to an emergency,
# lack of coverage for that specific procedure, or proximity). We cannot
# observe private-insurance patients who never use the public network at all.
#
# Refined question: among privately-insured patients who DO end up in the
# public network, is their severity profile lower than FONASA beneficiaries
# (suggesting they only use public care for minor/incidental cases) or
# similar/higher (suggesting they only arrive there for emergencies the
# private network could not resolve in time)?
#
# Result found: severity is consistently LOWER for PRIVATE in 2022-2024
# (gap +4.9 to +5.5pp), supporting the first interpretation. 2021 is an
# outlier, explained separately (see pandemic/elective-surgery note above).
#
# Key limitation to state in the report: this does NOT allow a full
# "public vs private system" complexity comparison - only a comparison of
# who, among those who do reach the public network, arrives with higher or
# lower severity depending on their insurance type.
# ---------------------------------------------------------------------------

In [5]:
NEEDED_COLS = ["PREVISION", "IR_29301_PESO", "IR_29301_SEVERIDAD", "IR_29301_MORTALIDAD", "TIPO_INGRESO"]

yearly_data = {}  # stores each year's cleaned dataframe for reuse

for year, (path, enc) in GRD_PATHS.items():
    valid, stats = load_and_clean_grd(year, path, enc)
    yearly_data[year] = valid
    print(f"{year}: loaded, {len(valid)} valid rows")

admission_results = []
for year, valid in yearly_data.items():
    pct_urgent = valid.groupby("insurance_group")["TIPO_INGRESO"].apply(
        lambda x: (x.str.upper() == "URGENCIA").mean() * 100
    )
    admission_results.append({
        "year": year,
        "pct_urgent_FONASA": pct_urgent.get("FONASA"),
        "pct_urgent_PRIVATE": pct_urgent.get("PRIVATE"),
    })

admission_summary = pd.DataFrame(admission_results).sort_values("year")
print(admission_summary)

2021: loaded, 703387 valid rows
2022: loaded, 777989 valid rows
2023: loaded, 845984 valid rows
2024: loaded, 873711 valid rows
   year  pct_urgent_FONASA  pct_urgent_PRIVATE
0  2021          64.220074            72.87902
1  2022          59.414604            64.91583
2  2023          60.477651            66.34073
3  2024          62.316541            69.28275
